In [1]:
import os
import sys

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir,"..","..",".."))

sys.path.append(project_root)

In [1]:
from src.utils.citibike_utils import get_trip_duration_mins
from src.utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map,lit

In [ ]:
catalog = dbutils.widgets.get("catalog")
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")

In [3]:
df = get_trip_duration_mins(spark,df,"started_at","ended_at","trip_duration_min")

In [4]:
df = timestamp_to_date_col(spark,df,"started_at","trip_start_date")

In [ ]:
# Extract the values first
pipeline_id_val = dbutils.widgets.get("pipelineid")
run_id_val = dbutils.widgets.get("runid")
task_id_val = dbutils.widgets.get("task_id")
processed_at_val = dbutils.widgets.get("processed_at")


# Apply transformations using the extracted variables
df = df.withColumn("metadata",
    create_map(
        lit("pipelineid"), lit(pipeline_id_val),
        lit("runid"), lit(run_id_val),
        lit("task_id"), lit(task_id_val),
        lit("processed_at"), lit(processed_at_val)
    )
)


In [ ]:
df.write.mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{catalog}.02_silver.jc_citibike")